In [ ]:
#importacion de librerias
from google.colab import files
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
#subida del archivo con los datos climatologicos
uploaded = files.upload()

Saving df_limpio.xlsx to df_limpio.xlsx


In [ ]:
#dataframe que guarda el archivo con los datos climatologicos
df = pd.read_excel("df_limpio.xlsx")

In [ ]:
#subida del archivo con los datos historicos de urgencias
uploaded = files.upload()

Saving urgencias-hospitalarias-atendidas_utf.csv to urgencias-hospitalarias-atendidas_utf.csv


In [ ]:
#data frame que guarda el archivo con los datos historicos de urgencias
df_final = pd.read_csv("urgencias-hospitalarias-atendidas_utf.csv", sep=";", encoding="latin1")

In [ ]:
df_final.head()

,Fecha de atención,Día de la semana,Hora,Nivel de triaje,Zona Básica de Salud,Ámbito de procedencia,Hospital,Área,Provincia,Edad,Sexo
0,22/01/2025,MIÉRCOLES,5:41,2,Z.B.S. Eras de Renueva,Urbano,C.A.U. León,León,León,52.0,Mujer
1,21/01/2025,MARTES,14:33,3,Z.B.S. Eras de Renueva,Urbano,C.A.U. León,León,León,14.0,Hombre
2,21/01/2025,MARTES,7:30,3,Z.B.S. Eras de Renueva,Urbano,C.A.U. León,León,León,68.0,Hombre
3,23/01/2025,JUEVES,0:33,4,Z.B.S. Eras de Renueva,Urbano,C.A.U. León,León,León,40.0,Mujer
4,25/01/2025,SÁBADO,17:52,3,Z.B.S. Eras de Renueva,Urbano,C.A.U. León,León,León,63.0,Hombre


In [ ]:
df.head()

,Día,T,TM,Tm,V,VM,PP,H,Mes,Year,Provincia,fecha
0,1,4.6,8.9,1.8,7.0,18.0,0.0,68,1,2024,Segovia,2024-01-01
1,2,8.5,12.4,2.8,10.8,37.8,0.0,58,1,2024,Segovia,2024-01-02
2,3,9.9,12.0,6.9,11.9,31.0,4.6,86,1,2024,Segovia,2024-01-03
3,4,7.6,10.5,6.1,8.5,22.0,5.4,87,1,2024,Segovia,2024-01-04
4,5,2.8,4.6,1.5,15.4,44.6,16.0,93,1,2024,Segovia,2024-01-05


In [ ]:
#guardado del campo Fecha de atención, que esta como string, en un nuevo campo fecha como tipo fecha
df_final['fecha'] = pd.to_datetime(df_final['Fecha de atención'])

/tmp/ipykernel_2834/2513191667.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_final['fecha'] = pd.to_datetime(df_final['Fecha de atención'])


In [ ]:
#cambio del formato fecha para formato fecha dia mes año
df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%Y')

In [ ]:
#creacion de un nuevo data frame combinando los datos de climatologica y urgencia haciendo como columna comun para la union provincia y fecha
df_merged = pd.merge(df_final, df, on=['Provincia','fecha'], how='left')

In [ ]:
#comprobacion de nulos
df_merged.isna().sum()

,0
Fecha de atención,0
Día de la semana,0
Hora,0
Nivel de triaje,0
Zona Básica de Salud,45931
Ámbito de procedencia,45931
Hospital,0
Área,0
Provincia,0
Edad,728


In [ ]:
#eliminado de nulos
df_merged = df_merged.dropna()

In [ ]:
#comprobado de nulos nuevamente
df_merged.isna().sum()

,0
Fecha de atención,0
Día de la semana,0
Hora,0
Nivel de triaje,0
Zona Básica de Salud,0
Ámbito de procedencia,0
Hospital,0
Área,0
Provincia,0
Edad,0


SACAR OTRA QUE REPRESENTE EL TIPO DE POBLACIÓN DE CADA PROVINCIA (ENVEJECIDA, JOVEN, NEUTRAL)

In [ ]:
df_merged.columns

Index(['Fecha de atención', 'Día de la semana', 'Hora', 'Nivel de triaje',
       'Zona Básica de Salud', 'Ámbito de procedencia', 'Hospital', 'Área',
       'Provincia', 'Edad', 'Sexo', 'fecha', 'Día', 'T', 'TM', 'Tm', 'V', 'VM',
       'PP', 'H', 'Mes', 'Year'],
      dtype='object')

In [ ]:
#eliminacion de columnas innecesarias
df_merged = df_merged.drop(columns=['Fecha de atención', 'Zona Básica de Salud', 'Área'])

In [ ]:
df_merged.columns

Index(['Día de la semana', 'Hora', 'Nivel de triaje', 'Ámbito de procedencia',
       'Hospital', 'Provincia', 'Edad', 'Sexo', 'fecha', 'Día', 'T', 'TM',
       'Tm', 'V', 'VM', 'PP', 'H', 'Mes', 'Year'],
      dtype='object')

In [ ]:
#creacion de un nuevo data frame con la agrupacion de pacientes segun provincia y fecha
pacientes_dia = df_merged.groupby(['Provincia','fecha']).size().reset_index(name='pacientes')
#creacion de un nuevo dataframe con la combinacion de los datos de urgencias con los datos climatologicos con los datos agrupados de pacientes, unido mediande provincia y fecha
df_modelo = pd.merge(pacientes_dia, df_merged, on=['Provincia','fecha'], how='left')
#creacion del campo target usando los pacientes agrupados segun provincia
df_modelo['target'] = df_modelo.groupby('Provincia')['pacientes'].shift(-1)
#creacion del campo pacientes ayer usando los pacientes agrupados segun provincia
df_modelo['pacientes_ayer'] = df_modelo.groupby('Provincia')['pacientes'].shift(1)
#orden de valores de proncia y fecha
df_modelo = df_modelo.sort_values(['Provincia','fecha'])

In [ ]:
#fd_modelo es nuetro data frame final
df_modelo

,Provincia,fecha,pacientes,Día de la semana,Hora,Nivel de triaje,Ámbito de procedencia,Hospital,Edad,Sexo,...,TM,Tm,V,VM,PP,H,Mes,Year,target,pacientes_ayer
0,Burgos,2024-01-01,188,LUNES,1:52,3,Urbano,C.A.U. Burgos,39.0,Mujer,...,6.5,1.0,17.0,27.8,1.27,84.0,1.0,2024.0,188.0,NaN
1,Burgos,2024-01-01,188,LUNES,17:57,2,Urbano,C.A.U. Burgos,80.0,Hombre,...,6.5,1.0,17.0,27.8,1.27,84.0,1.0,2024.0,188.0,188.0
2,Burgos,2024-01-01,188,LUNES,2:59,2,Urbano,C.A.U. Burgos,58.0,Mujer,...,6.5,1.0,17.0,27.8,1.27,84.0,1.0,2024.0,188.0,188.0
3,Burgos,2024-01-01,188,LUNES,13:28,3,Urbano,C.A.U. Burgos,55.0,Mujer,...,6.5,1.0,17.0,27.8,1.27,84.0,1.0,2024.0,188.0,188.0
4,Burgos,2024-01-01,188,LUNES,10:45,4,Urbano,C.A.U. Burgos,66.0,Mujer,...,6.5,1.0,17.0,27.8,1.27,84.0,1.0,2024.0,188.0,188.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
790993,Ávila,2025-08-31,79,DOMINGO,16:55,4,Urbano,C.A. Ávila,68.0,Mujer,...,0.0,0.0,0.0,0.0,0.00,0.0,8.0,2025.0,79.0,79.0
790994,Ávila,2025-08-31,79,DOMINGO,23:35,4,Urbano,C.A. Ávila,69.0,Mujer,...,0.0,0.0,0.0,0.0,0.00,0.0,8.0,2025.0,79.0,79.0
790995,Ávila,2025-08-31,79,DOMINGO,17:55,4,Urbano,C.A. Ávila,69.0,Mujer,...,0.0,0.0,0.0,0.0,0.00,0.0,8.0,2025.0,79.0,79.0
790996,Ávila,2025-08-31,79,DOMINGO,6:45,Desconocido,Urbano,C.A. Ávila,39.0,Mujer,...,0.0,0.0,0.0,0.0,0.00,0.0,8.0,2025.0,79.0,79.0


In [ ]:
#rellenado de nulos de pacientes ayer con los pacientes
df_modelo['pacientes_ayer'] = df_modelo['pacientes_ayer'].fillna(df_modelo['pacientes'])

In [ ]:
#borrado de nulos
df_modelo = df_modelo.dropna()

In [ ]:
#visualizado de nulos
df_modelo.isna().sum()

,0
Provincia,0
fecha,0
pacientes,0
Día de la semana,0
Hora,0
Nivel de triaje,0
Ámbito de procedencia,0
Hospital,0
Edad,0
Sexo,0


CAMBIOS NUEVOS

In [ ]:
df_modelo.head()

,Provincia,fecha,pacientes,Día de la semana,Hora,Nivel de triaje,Ámbito de procedencia,Hospital,Edad,Sexo,...,TM,Tm,V,VM,PP,H,Mes,Year,target,pacientes_ayer
0,Burgos,2024-01-01,188,LUNES,1:52,3,Urbano,C.A.U. Burgos,39.0,Mujer,...,6.5,1.0,17.0,27.8,1.27,84.0,1.0,2024.0,188.0,188.0
1,Burgos,2024-01-01,188,LUNES,17:57,2,Urbano,C.A.U. Burgos,80.0,Hombre,...,6.5,1.0,17.0,27.8,1.27,84.0,1.0,2024.0,188.0,188.0
2,Burgos,2024-01-01,188,LUNES,2:59,2,Urbano,C.A.U. Burgos,58.0,Mujer,...,6.5,1.0,17.0,27.8,1.27,84.0,1.0,2024.0,188.0,188.0
3,Burgos,2024-01-01,188,LUNES,13:28,3,Urbano,C.A.U. Burgos,55.0,Mujer,...,6.5,1.0,17.0,27.8,1.27,84.0,1.0,2024.0,188.0,188.0
4,Burgos,2024-01-01,188,LUNES,10:45,4,Urbano,C.A.U. Burgos,66.0,Mujer,...,6.5,1.0,17.0,27.8,1.27,84.0,1.0,2024.0,188.0,188.0


In [ ]:
df_copia = df_modelo

In [ ]:
df_modelo = df_copia

In [ ]:
#cambio de nombres de campos
df_modelo = df_modelo.rename(columns={
    "Provincia": "provincia",
    "Día de la semana": "dia_semana",
    "Hora": "hora",
    "Nivel de triaje": "nivel_triaje",
    "Ámbito de procedencia": "ambito_procedencia",
    "Hospital": "hospital",
    "Edad": "edad",
    "Sexo": "sexo",
    "Día": "dia",
    "Mes": "mes",
    "Year": "year",
    "target": "target_pacientes"
})

In [ ]:
#cambios de tipos de los campos
df_modelo["edad"] = df_modelo["edad"].astype(int)
df_modelo["dia"] = df_modelo["dia"].astype(int)
df_modelo["mes"] = df_modelo["mes"].astype(int)
df_modelo["year"] = df_modelo["year"].astype(int)
df_modelo["target_pacientes"] = df_modelo["target_pacientes"].astype(int)
df_modelo["pacientes_ayer"] = df_modelo["pacientes_ayer"].astype(int)

In [ ]:
#cambio de orden de columna
#creacion de columna axuliar con pop, que retiramos la columna de su data frame y se guarda en esta
columna_auxiliar = df_modelo.pop("target_pacientes")
#se crea nuevamente la columna quitada con los datos de la columna axuliar, asi se queda a la derecha en el data frame
df_modelo["target_pacientes"] = columna_auxiliar
#borrado de pacientes
df_modelo = df_modelo.drop('pacientes', axis=1)

In [ ]:
#creacion columna fecha hora
columna_auxiliar["hora"] = df_modelo["hora"]
#se añade los segundos para que el formato quede como hh:mm:ss y no explote con el deltatime
columna_auxiliar["hora"] = columna_auxiliar['hora'].str.pad(5, side='left', fillchar='0') + ':00'
#cambio a tipo hora
columna_auxiliar["hora"] = pd.to_timedelta(columna_auxiliar["hora"])
#creacion de la columna en el data frame de datos
df_modelo["fecha_hora"] = df_modelo["fecha"] + columna_auxiliar["hora"]
df_modelo.head()

,provincia,fecha,dia_semana,hora,nivel_triaje,ambito_procedencia,hospital,edad,sexo,dia,...,Tm,V,VM,PP,H,mes,year,pacientes_ayer,target_pacientes,fecha_hora
0,Burgos,2024-01-01,LUNES,1:52,3,Urbano,C.A.U. Burgos,39,Mujer,1,...,1.0,17.0,27.8,1.27,84.0,1,2024,188,188,2024-01-01 01:52:00
1,Burgos,2024-01-01,LUNES,17:57,2,Urbano,C.A.U. Burgos,80,Hombre,1,...,1.0,17.0,27.8,1.27,84.0,1,2024,188,188,2024-01-01 17:57:00
2,Burgos,2024-01-01,LUNES,2:59,2,Urbano,C.A.U. Burgos,58,Mujer,1,...,1.0,17.0,27.8,1.27,84.0,1,2024,188,188,2024-01-01 02:59:00
3,Burgos,2024-01-01,LUNES,13:28,3,Urbano,C.A.U. Burgos,55,Mujer,1,...,1.0,17.0,27.8,1.27,84.0,1,2024,188,188,2024-01-01 13:28:00
4,Burgos,2024-01-01,LUNES,10:45,4,Urbano,C.A.U. Burgos,66,Mujer,1,...,1.0,17.0,27.8,1.27,84.0,1,2024,188,188,2024-01-01 10:45:00


In [ ]:
df_modelo.dtypes

,0
provincia,object
fecha,datetime64[ns]
dia_semana,object
hora,object
nivel_triaje,object
ambito_procedencia,object
hospital,object
edad,int64
sexo,object
dia,int64


In [ ]:
#guardado de datos como csv
df_modelo.to_csv("df_modelo_target.csv", index=False)